# 3 — Canary Insertion

Generate canary strings and produce **one augmented dataset per frequency level**.
Each dataset contains the 10k base training posts plus a single canary repeated N times.

Canary format: `"My patient ID is XXXXXX"` (6-digit secret, unique per frequency level).

| Canary | Secret | Frequency | Dataset saved |
|--------|--------|-----------|---------------|
| C1 | generated below |  1× | `train_ds_canary_freq1` |
| C2 | generated below |  5× | `train_ds_canary_freq5` |
| C3 | generated below | 10× | `train_ds_canary_freq10` |
| C4 | generated below | 50× | `train_ds_canary_freq50` |

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets

## 3. Imports & Config

In [ ]:
import os
import json
import random
from datasets import load_from_disk, Dataset, concatenate_datasets
from transformers import GPT2TokenizerFast

# ── Config ─────────────────────────────────────────────────────────────────────
CANARY_PREFIX = 'My patient ID is '
CANARY_FREQS  = [1, 5, 10, 50]   # insertion frequencies to sweep
SEED_CANARY   = 2412
MAX_LENGTH    = 128

## 4. Generate Canary Secrets

In [ ]:
random.seed(SEED_CANARY)

# One unique 6-digit secret per frequency level
secrets = [str(random.randint(100000, 999999)) for _ in CANARY_FREQS]

canaries = [
    {
        'id':      f'C{i+1}',
        'prefix':  CANARY_PREFIX,
        'secret':  s,
        'text':    CANARY_PREFIX + s,
        'freq':    f,
        'dataset': f'train_ds_canary_freq{f}',
    }
    for i, (s, f) in enumerate(zip(secrets, CANARY_FREQS))
]

print('Generated canaries (seed=2412):')
print(f'{"ID":<5} {"Freq":>6}x   Text')
print('-' * 45)
for c in canaries:
    print(f"{c['id']:<5} {c['freq']:>6}x   \"{c['text']}\"")

## 5. Load Tokenizer & Base Training Data

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(f'{PROJECT_DIR}/data/tokenizer')
train_ds  = load_from_disk(f'{PROJECT_DIR}/data/train_ds')

print(f'Tokenizer vocab size: {len(tokenizer)}')
print(f'Base train size:      {len(train_ds)}')
print(f'Columns:              {train_ds.column_names}')

## 6. Build One Augmented Dataset per Frequency

In [ ]:
def tokenize_text(text):
    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )

augmented_datasets = {}

for c in canaries:
    enc = tokenize_text(c['text'])

    # Repeat the same canary c['freq'] times
    canary_rows = {
        'input_ids':      [enc['input_ids']]      * c['freq'],
        'attention_mask': [enc['attention_mask']] * c['freq'],
    }
    canary_ds = Dataset.from_dict(canary_rows)

    augmented = concatenate_datasets([train_ds, canary_ds])
    augmented_datasets[c['dataset']] = augmented

    # Sanity-check: decode first canary row
    decoded = tokenizer.decode(enc['input_ids'], skip_special_tokens=True).strip()
    print(f"{c['id']}  freq={c['freq']:>2}x  size={len(augmented)}  canary: \"{decoded}\"")

## 7. Save to Drive

In [ ]:
print('Saving augmented datasets...')
for name, ds in augmented_datasets.items():
    path = f'{PROJECT_DIR}/data/{name}'
    ds.save_to_disk(path)
    print(f'  saved {name} ({len(ds)} examples) → {path}')

# Canary metadata (used by 5_evaluate.ipynb)
canaries_path = f'{PROJECT_DIR}/data/canaries.json'
with open(canaries_path, 'w') as f:
    json.dump(canaries, f, indent=2)
print(f'\nCanary metadata saved to: {canaries_path}')

## 8. Summary

In [ ]:
print('=== Canary Insertion Summary ===')
print(f'Seed:          {SEED_CANARY}')
print(f'Canary prefix: "{CANARY_PREFIX}"')
print(f'Base train:    {len(train_ds)} examples')
print()
print(f'{"Canary":<5}  {"Freq":>5}x  {"Secret":<10}  Dataset')
print('-' * 55)
for c in canaries:
    print(f"{c['id']:<5}  {c['freq']:>5}x  {c['secret']:<10}  {c['dataset']}")
print()
print('Each dataset = base train + one canary repeated N times.')
print('Models trained on these datasets are directly comparable across frequencies.')

---
**Next:** Run `4_dp_train.ipynb` to fine-tune with DP-SGD (Opacus) on each augmented dataset.